# Ta campaign

Metal-center transfer from Cu3VS4 to Cu3TaS4. CuI is held constant; VO(acac)2 is replaced by TaCl5. Data in `data_Ta/`.

Nine matched vanadium process conditions were run with TaCl5, then the closed loop targeted 32.5 nm. The metal descriptor is oxophilicity. Settings are set in the first code cell.


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve() / "src"))

import importlib
import config as _cfg
importlib.reload(_cfg)

# Ta campaign settings. Applied here so src/config.py does not need to be edited.
# Metal-center transfer: CuI held constant; VO(acac)2 replaced by TaCl5.
_cfg.CURRENT_PRECURSORS["Cu_precursor"] = "CuI"
_cfg.CURRENT_PRECURSORS["Metal_Precursor"] = "TaCl5"
_cfg.TRANSFER_MODE["enabled"] = True
_cfg.TRANSFER_MODE["vary_cu_precursor"] = False
_cfg.TRANSFER_MODE["vary_metal_precursor"] = True
_cfg.TRANSFER_MODE["target_cu_precursor"] = "CuI"
_cfg.TRANSFER_MODE["target_metal_precursor"] = "TaCl5"

# Enable the metal-axis descriptors used in this campaign.
_cfg.ENHANCED_FEATURE_CONFIG["Cu_precursor_hardness"] = False
_cfg.ENHANCED_FEATURE_CONFIG["Cu_hsab_mismatch"] = False
_cfg.ENHANCED_FEATURE_CONFIG["Metal_ionic_potential"] = True
_cfg.ENHANCED_FEATURE_CONFIG["Metal_oxophilicity"] = True
_cfg.TRANSFER_FEATURES = list(_cfg.SYNTHESIS_FEATURES) + ["Metal_ionic_potential", "Metal_oxophilicity"]
for _name in ("features", "optimizer", "selfvalidating", "visualization", "diagnostics", "experiment_store"):
    if _name in sys.modules:
        importlib.reload(sys.modules[_name])

from selfvalidating import SelfValidatingOptimizer
from config import COLORS, PUBLICATION_STYLE

from visualization import (
    plot_recommendation_history,
    plot_parity,
    plot_calibration,
    plot_error_learning_progress,
    plot_error_correction_impact,
    plot_target_achievement,
    plot_feature_importance,
    plot_acquisition_feature_importance,
    plot_precursor_paired_comparison,
    plot_classifier_calibration,
    plot_collinearity_heatmap,
    plot_dataset_quality_dashboard,
    plot_loo_residuals,
    plot_property_correlations,
    plot_acquisition_slice,
    plot_recommendation_regret,
    plot_response_surface,
    plot_classification_surface,
    plot_classification_facets,
    plot_loo_parity,
    plot_feasibility_landscape,
    plot_squareness_binning,
    plot_acquisition_decomposition,
    plot_bias_correction_arrows,
    plot_bin_probability_facets_all,
    plot_bin_probability_facets_cuv,
    plot_bo_trajectory,
    plot_bo_trajectory_by_size,
    plot_optimization_progress,
)

from diagnostics import (
    compare_feature_modes,
    detect_extrapolation,
    diagnose_collinearity,
    print_model_assessment,
    display_recommendations_table,
    get_optimization_convergence_summary,
    print_optimization_convergence_summary,
    print_optimization_statistics,
    compute_optimization_statistics,
)

DATA_DIR = Path("..") / "data_Ta"
OUTPUT_DIR = Path("..") / "outputs_Ta"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(PUBLICATION_STYLE)

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', message='.*lbfgs failed to converge.*')

np.random.seed(42)

print("All imports successful")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 1. Initialize

Load experiments, fit GPs and classifiers, and set up the error learner.


In [ ]:
# Ta campaign: pool CuI/VO(acac)2 history with the 9-point TaCl5 seed set.
# CSVs are imported only if experiments.json is empty.
optimizer = SelfValidatingOptimizer(
    data_dir=DATA_DIR,
    initialize_from_csv=True,
    initial_csv_imports=[
        {'path': DATA_DIR / 'CuI_transfer_data.csv', 'cu_precursor': 'CuI', 'metal_precursor': 'VO(acac)2'},
        {'path': DATA_DIR / 'Cu3TaS4_initial.csv', 'cu_precursor': 'CuI', 'metal_precursor': 'TaCl5'},
    ],
    feature_mode='synthesis',
)

optimizer.print_status()


## 1b. Diagnostics

LOO-CV, collinearity (VIF), classifier calibration, and sample-size checks.


In [ ]:
diagnostics = optimizer.full_diagnostics()

# Optional feature-mode comparison (re-fits each mode — ~2-3 min):
comparison_df = optimizer.compare_feature_modes(
    modes=['raw', 'chemical', 'hybrid', 'synthesis', 'transfer']
)

# Synthesis->descriptor predictive-skill table for the transfer-evidence figure.
# Size uses LOO R2; PhasePure/IsCubic use out-of-sample Brier skill scores.
# Refits GPs + classifier CV (~5-6 min); cached on the optimizer after first run.
skill_df = optimizer.compute_transfer_skill_table(
    modes=('synthesis', 'transfer'),
    classifier_targets=['PhasePure', 'IsCubic'],
)


## 1c. Transfer evidence

Matched process conditions between VO(acac)2 and TaCl5, skill with vs without metal descriptors, and size LOO parity coloured by metal precursor.


In [ ]:
# Reuse comparison_df / skill_df from the diagnostics cell if they were already computed.
fig = plot_precursor_paired_comparison(
    optimizer,
    comparison_df=comparison_df if 'comparison_df' in dir() else None,
    skill_df=skill_df if 'skill_df' in dir() else None,
    precursors=['VO(acac)2', 'TaCl5'],
    precursor_col='Metal_precursor',
    color_overrides={'VO(acac)2': '#C9B3D9', 'TaCl5': '#4A2A75'},
)
if fig:
    plt.savefig(OUTPUT_DIR / 'precursor_transfer_evidence.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Reliability diagrams for the feasibility classifiers.
fig_cal = plot_classifier_calibration(optimizer)
if fig_cal:
    plt.savefig(OUTPUT_DIR / "classifier_calibration.png", dpi=150, bbox_inches='tight')
    plt.show()

# Pairwise feature correlation heatmap.
fig_corr = plot_collinearity_heatmap(optimizer)
if fig_corr:
    plt.savefig(OUTPUT_DIR / "feature_correlations.png", dpi=150, bbox_inches='tight')
    plt.show()


## Dataset quality

Design-space coverage, outcome distributions, and LOO-CV R2 by feature mode.


In [ ]:
fig_dashboard = plot_dataset_quality_dashboard(
    optimizer,
    figsize=(14, 10),
    save_path=OUTPUT_DIR / "dataset_quality_dashboard.png",
)

if fig_dashboard:
    plt.show()
    print(f"\nDashboard saved to: {OUTPUT_DIR / 'dataset_quality_dashboard.png'}")


## 2. Generate recommendations

Set a target size and a squareness bin:

- `highly_cubic`: cubic, squareness at or above the Otsu threshold (0.810)
- `poorly_cubic`: cubic, squareness below the threshold
- `multipod`: non-cubic


In [ ]:
# TARGET_SIZE = 32.5
# SIZE_TOLERANCE = 2.0
# SQUARENESS_BIN = 'highly_cubic'  # 'highly_cubic' | 'poorly_cubic' | 'multipod'

# recommendations = optimizer.recommend(
#    target_size=TARGET_SIZE,
#    size_tol=SIZE_TOLERANCE,
#    squareness_bin=SQUARENESS_BIN,
#    seed=42,
#)
# display_recommendations_table(recommendations)


## 3. Pending recommendations

Recommendations waiting on lab results.


In [ ]:
pending = optimizer.get_pending_recommendations()
display(pending)


## 4. Complete recommendations

Log measured results once the synthesis is done.


In [ ]:
# Example: log measured results for a completed recommendation.

# errors = optimizer.complete_recommendation(
#    rec_id='REC_008',         # recommendation ID
#    Size=34.089,              # measured size (nm)
#    CV=0.389,                 # measured CV
#    Squareness=0.825,         # measured squareness
#    HasProduct=1,             # 1 if product formed
#    PhasePure=1,              # 1 if phase pure
#    Polymorph='cubic',        # 'cubic', 'multipod', etc.
# )

# print_model_assessment(optimizer)


## 5. Results

Recommendation history, trajectory toward the target, parity, and achieved sizes.


In [ ]:
fig = plot_recommendation_history(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'recommendation_history.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_bo_trajectory(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'bo_trajectory.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_bo_trajectory_by_size(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / "bo_trajectory_by_size.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
fig = plot_target_achievement(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'target_achievement.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_parity(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'parity_plots_recommendations.png', dpi=300, bbox_inches='tight')
    plt.show()


## 6. Model quality

Residual checks, LOO-CV parity, calibration, and feature importance.


In [ ]:
fig = plot_calibration(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'calibration_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_loo_residuals(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'loo_residuals.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_loo_parity(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'loo_parity.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_feature_importance(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Feature importance for the three BO acquisition components (not the diagnostic CV/Squareness GPs).
fig = plot_acquisition_feature_importance(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'acquisition_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()


## 7. Error correction

Bias estimates, correction impact, and prediction error over the campaign.


In [ ]:
print_model_assessment(optimizer)


In [ ]:
fig = plot_error_learning_progress(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'learning_progress.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_error_correction_impact(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'error_correction_impact.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Requires the ErrorLearner to be fitted (≥10 completed recommendations).
fig = plot_bias_correction_arrows(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'bias_correction_arrows.png', dpi=300, bbox_inches='tight')
    plt.show()


## 8. Design space

Where the experiments sit in parameter space, and how feasibility, morphology, and acquisition vary.


In [ ]:
fig = plot_property_correlations(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'property_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_feasibility_landscape(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'feasibility_landscape.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig = plot_squareness_binning(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'squareness_binning.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Acquisition decomposition: P(size band) × P(feasible) × P(bin).
# Target size, tolerance, and morphology bin.
fig = plot_acquisition_decomposition(
    optimizer,
    target_size=15.0,
    size_tol=2.0,
    squareness_bin='highly_cubic',
)
if fig:
    plt.savefig(OUTPUT_DIR / 'acquisition_decomposition.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Optional 2-D acquisition slice.
# fig = plot_acquisition_slice(
#     optimizer,
#     target_size=20.0,
#     size_tol=2.5,
#     squareness_bin='highly_cubic',
#     param1='Temp',
#     param2=None,
# )
# if fig:
#     plt.savefig(OUTPUT_DIR / 'acquisition_slice.png', dpi=300, bbox_inches='tight')
#     plt.show()


### Response and classification surfaces

Top three features by gradient importance; remaining features at cubic-training medians.


In [ ]:
for resp in ['Size', 'CV', 'Squareness', 'Feasibility']:
    fig = plot_response_surface(optimizer, response=resp, n_grid=25)
    if fig:
        plt.savefig(
            OUTPUT_DIR / f'response_surface_{resp.lower()}.png',
            dpi=300,
            bbox_inches='tight',
        )
        plt.show()


In [ ]:
fig = plot_classification_surface(optimizer, n_grid=25)
if fig:
    plt.savefig(OUTPUT_DIR / 'classification_surface_squareness.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# 3×3 facet grid per remaining feature (top-2 axes × slice dimension).
figs = plot_bin_probability_facets_all(optimizer, n_grid=50)
for i, fig in enumerate(figs or []):
    if fig:
        plt.savefig(
            OUTPUT_DIR / f'bin_probability_facets_all_{i}.png',
            dpi=300,
            bbox_inches='tight',
        )
        plt.show()


In [ ]:
# 3×3 facets with Cu/V ratio fixed as the column slice.
fig = plot_bin_probability_facets_cuv(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'bin_probability_facets_cuv.png', dpi=300, bbox_inches='tight')
    plt.show()


## 8a. Optimization progress

Running-best CV and squareness compared with the initial dataset.


In [ ]:
fig = plot_optimization_progress(optimizer)
if fig:
    fig.savefig(OUTPUT_DIR / 'optimization_progress.png', dpi=300, bbox_inches='tight')
    plt.show()


## 8b. Statistical tests

Hit rate, Mann-Whitney vs baseline, GP calibration, and prediction-error trend.


In [ ]:
stats_results = print_optimization_statistics(optimizer)

# Optional: export the summary table.
# stats_results["summary_table"].to_csv(
#     OUTPUT_DIR / "optimization_statistics.csv",
#     index=False,
# )


## 9. Convergence

Regret curves and prediction quality vs targets.


In [ ]:
fig = plot_recommendation_regret(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'recommendation_regret.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Need at least 2 completed recommendations to plot regret.')


In [ ]:
print_optimization_convergence_summary(optimizer, last_n=5)
